# Đóng gói thư viện vLLM cho Qwen offline (Python 3.12)
Chạy trên Kaggle (Internet bật, CPU), lưu đầu ra thành Dataset.

In [ ]:
import json, os, subprocess, sys
from pathlib import Path

OUT = Path("/kaggle/working/thu_vien")
OUT.mkdir(parents=True, exist_ok=True)
YEU_CAU = ["vllm==0.32.0", "pandas", "pyarrow", "huggingface_hub"]     # cùng bản vllm với bộ 1-3
CHUOI = " ".join(f"'{y}'" for y in YEU_CAU)

def sh(c, tail=3000):
    print(">>", c[:300], flush=True)
    r = subprocess.run(c, shell=True, text=True, capture_output=True)
    print((r.stdout or "")[-tail:], (r.stderr or "")[-tail:], flush=True)
    return r.returncode

# Lần 1 và 2 (26/10/2026) lỗi: `pip download --platform ...` chạy trên Python 3.13 của máy chủ Kaggle nhưng điều kiện môi trường (numpy 2.4, ...)
# vẫn tính theo 3.13 -> bộ wheel không khớp Python 3.12 của máy đích. Cách đúng: dựng Python 3.12 thật rồi tải bằng pip của nó.
print("glibc:", subprocess.run("ldd --version | head -1", shell=True, capture_output=True, text=True).stdout.strip(), flush=True)
sh("which uv || pip install -q uv")
assert sh("uv python install 3.12") == 0
assert sh("uv venv /tmp/py312 --python 3.12 --seed") == 0
PY = "/tmp/py312/bin/python"
sh(f"{PY} --version; {PY} -m pip --version")
assert sh(f"{PY} -m pip download --no-cache-dir -d {OUT} --only-binary=:all: {CHUOI}") == 0, "pip download lỗi: xem log phía trên"
ws = sorted(OUT.glob("*"))
tong = sum(p.stat().st_size for p in ws)
print(f"{len(ws)} file, {tong / 2**30:.2f} GiB")
for p in sorted(ws, key=lambda p: -p.stat().st_size)[:8]:
    print(f"  {p.stat().st_size / 2**20:9.1f} MiB  {p.name}")
assert tong < 19 * 2**30, "vượt giới hạn 20 GB của /kaggle/working"
# máy đích dùng glibc 2.35: không được có wheel đòi manylinux_2_36 trở lên
import re
cao = [p.name for p in ws for m in re.finditer(r"manylinux_2_(\d+)", p.name) if int(m.group(1)) > 35]
assert not cao, f"có wheel đòi glibc > 2.35: {cao[:5]}"
# kiểm THẬT: cài offline (không Internet) vào một môi trường Python 3.12 sạch, rồi kiểm phụ thuộc
assert sh("uv venv /tmp/env_kiem --python 3.12") == 0
rc2 = sh(f"uv pip install --no-index --find-links {OUT} --python /tmp/env_kiem/bin/python {CHUOI}", tail=4000)
rc3 = sh("uv pip check --python /tmp/env_kiem/bin/python") if rc2 == 0 else 1
sh("uv pip list --python /tmp/env_kiem/bin/python | grep -i -E '^(vllm|torch|numpy|pandas|pyarrow|huggingface|transformers|triton|flashinfer) '")
(OUT / "MANIFEST.json").write_text(json.dumps({"yeu_cau": YEU_CAU, "python": "3.12", "so_file": len(ws), "tong_byte": tong, "cai_offline_dat": rc2 == 0,
                                               "pip_check_dat": rc3 == 0, "files": [{"ten": p.name, "byte": p.stat().st_size} for p in ws]},
                                              indent=1), encoding="utf-8")
assert rc2 == 0, "cài offline KHÔNG đạt: thiếu gói trong bộ wheel, xem log"
assert rc3 == 0, "uv pip check báo xung đột phụ thuộc, xem log"
print("XONG: bộ thư viện cài offline được trên Python 3.12.")
